# Module 4 : Text mining, applications

*Mercredi 24 juin 2026, 9h-12h · Intervenant : Corentin Vande Kerckhove*

**CONSEIL :** Sauvegarde une copie de ce notebook dans ton Drive avant de commencer !

**Objectifs d'apprentissage**
- Enchaîner **quatre applications** concrètes du text mining, de la plus simple à la plus avancée
- Décrire un corpus avec un **nuage de mots** et des **mots-clés** distinctifs
- Mesurer un **sentiment** avec un modèle pré-entraîné
- Entraîner une **classification supervisée** (ML) et lire ses résultats
- Faire de la **recherche sémantique** par embeddings

**Pré-requis :** Module 3 (concepts du text mining).

**Paquets requis :** `scikit-learn`, `transformers`, `sentence-transformers`, `wordcloud`, `matplotlib`, `pandas`

## Le corpus : critiques de films notées (Allociné)

On travaille sur un échantillon de **critiques de films** rédigées sur Allociné, chacune étiquetée **positif** ou **négatif** d'après la note laissée par l'internaute. Un corpus idéal pour le text mining : du texte libre, en français, avec une **étiquette** qui permet d'illustrer aussi bien les méthodes descriptives que l'apprentissage supervisé.

| Fichier | Lignes | Équilibre |
|---|---|---|
| `critiques_films_train.csv` | 3 000 | 1 500 positifs / 1 500 négatifs |
| `critiques_films_test.csv` | 1 000 | 500 positifs / 500 négatifs |

![Salle de cinéma](https://raw.githubusercontent.com/mickaeltemporao/lillelms/refine-notebooks-j2-j3-matin/ressources/j3/1_matin/img/m4-opener.jpg)

*Running example : analyser des critiques de films notées. Photo : salle de cinéma, Studio Sarah Lou, CC BY 2.0.*

## Setup

In [ ]:
# Sur Colab : exécute cette cellule pour installer les paquets (déjà installés sur un poste local)
!pip install -q scikit-learn transformers sentence-transformers wordcloud matplotlib pandas

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Les données se chargent **toutes seules**, sans aucun fichier à uploader : on lit la copie hébergée si elle est disponible, sinon on reconstruit l'échantillon **directement depuis la source** ([Allociné sur Hugging Face](https://huggingface.co/datasets/tblard/allocine)).

In [ ]:
import io, requests

def telecharger_allocine(split, n_par_classe):
    url = (f"https://huggingface.co/datasets/tblard/allocine/resolve/main/"
           f"allocine/{split}-00000-of-00001.parquet")
    brut = pd.read_parquet(io.BytesIO(requests.get(url, timeout=180).content))
    parts = [g.sample(min(n_par_classe, len(g)), random_state=42)
             for _, g in brut.groupby("label")]
    out = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)
    out = out.rename(columns={"review": "text", "label": "polarite"})
    out["polarite"] = out["polarite"].map({0: "négatif", 1: "positif"})
    return out[["text", "polarite"]]

def charger(split, n_par_classe):
    # On essaie d'abord l'URL GitHub (fonctionne sur Colab), puis le fichier local,
    # et en dernier recours on reconstruit l'échantillon depuis Hugging Face.
    url = ("https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/"
           f"data/critiques_films/critiques_films_{split}.csv")
    local = f"../../../data/critiques_films/critiques_films_{split}.csv"
    try:
        df = pd.read_csv(url)
        print(f"Données chargées depuis : GitHub (raw)")
        return df
    except Exception:
        pass
    if os.path.exists(local):
        print(f"Données chargées depuis : fichier local")
        return pd.read_csv(local)
    print(f"Reconstruction du split '{split}' depuis Hugging Face...")
    df = telecharger_allocine(split, n_par_classe)
    print(f"Données chargées depuis : Hugging Face (reconstruction)")
    return df

train = charger("train", 1500)
test = charger("test", 500)

print(f"Train : {len(train):,} | Test : {len(test):,}")
train.head(3)

On aura besoin d'une liste de **stopwords français** (mots outils à ignorer). `scikit-learn` n'en fournit pas, on en définit une courte ici.

In [ ]:
STOP_FR = set('''
au aux avec ce ces dans de des du elle en et eux il je la le les leur lui ma
mais me même mes moi mon ne nos notre nous on ou par pas pour qu que qui sa se
ses son sur ta te tes toi ton tu un une vos votre vous c d j l à m n s t y été
être ai as avait avais avons ont est sont a als cette cet plus moins très bien
fait film films très tout tous toute toutes plus comme aussi car donc alors
sans sous entre vers chez ni leurs cela ça là où dont quand
'''.split())
print(len(STOP_FR), "stopwords")

## 4.1 Application 1 : Nuage de mots et mots-clés distinctifs

*La plus simple : descriptive, sans aucun modèle.*

**À quoi ça sert ?** C'est la première chose qu'on fait face à un corpus qu'on ne connaît pas : voir en un coup d'œil de quoi il parle, et ce qui **distingue deux groupes**. En SHS : comparer le vocabulaire de deux partis dans des débats parlementaires, repérer ce qui sépare les réponses ouvertes de deux profils d'enquêtés, ou résumer des milliers d'avis clients. Rapide, lisible, parfait pour une première intuition ou une figure d'article.

**Comment ?** On compte les mots et on regarde lesquels distinguent le mieux les bonnes des mauvaises critiques. Le **log-odds** compare la fréquence d'un mot chez les positifs et chez les négatifs : très positif d'un côté, très négatif de l'autre.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer(min_df=10, stop_words=list(STOP_FR))
X = vec.fit_transform(train["text"])
mots = np.array(vec.get_feature_names_out())

pos = np.asarray(X[(train["polarite"] == "positif").values].sum(axis=0)).ravel()
neg = np.asarray(X[(train["polarite"] == "négatif").values].sum(axis=0)).ravel()

# log-odds lissé : positif si > 0, négatif si < 0.
# Le +1 (lissage) évite de diviser par zéro ou de prendre le log de zéro
# pour un mot qui n'apparaît que dans une seule des deux classes.
logodds = np.log((pos + 1) / (pos.sum() + 1)) - np.log((neg + 1) / (neg.sum() + 1))
ordre = logodds.argsort()

print("Mots plutôt POSITIFS :", ", ".join(mots[ordre[::-1][:15]]))
print("Mots plutôt NÉGATIFS :", ", ".join(mots[ordre[:15]]))

Et visuellement, avec un **nuage de mots** par polarité :

In [ ]:
from wordcloud import WordCloud

def nuage(polarite, ax):
    texte = " ".join(train[train["polarite"] == polarite]["text"])
    wc = WordCloud(width=500, height=350, background_color="white",
                   stopwords=STOP_FR, colormap="viridis").generate(texte)
    ax.imshow(wc); ax.axis("off"); ax.set_title(f"Critiques {polarite}s")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
nuage("positif", axes[0])
nuage("négatif", axes[1])
plt.show()

### Hack Time

- Le nuage de mots est dominé par des mots fréquents mais peu informatifs ? Enrichis `STOP_FR` et relance.
- Affiche les 30 mots les plus négatifs au lieu de 15.

**Défi optionnel :** refais l'analyse sur les **bigrammes** (paires de mots) en passant `ngram_range=(2, 2)` au `CountVectorizer`, et regarde si les expressions distinctives sont plus parlantes que les mots seuls.

In [ ]:
# Votre code ici

## 4.2 Application 2 : Analyse de sentiment (modèle pré-entraîné)

*On monte d'un cran : on utilise un modèle déjà entraîné, en boîte noire.*

**À quoi ça sert ?** Mettre une **mesure** (positif / négatif) sur des volumes de texte qu'on ne pourra jamais lire à la main : suivre la tonalité de tweets sur une réforme au fil du temps, mesurer la satisfaction dans des réponses ouvertes d'enquête, comparer la charge émotionnelle d'articles de presse selon le média. On obtient une variable quantitative à croiser ensuite avec le reste de ses données.

**Comment ?** La librairie `transformers` charge en une ligne un modèle multilingue qui prédit un sentiment **sans qu'on l'ait entraîné**. On l'applique à un échantillon du test et on compare ses prédictions aux vraies étiquettes.

![Schéma analyse de sentiment](https://raw.githubusercontent.com/mickaeltemporao/lillelms/refine-notebooks-j2-j3-matin/ressources/j3/1_matin/img/m4-sentiment.png)

*Analyse de sentiment : le modèle attribue une note de polarité à un texte.*

In [ ]:
from transformers import pipeline

analyseur = pipeline(
    "sentiment-analysis",
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student",
)

echantillon = test.sample(200, random_state=42).copy()
# truncation=True : les critiques très longues qui dépassent la taille maximale
# du modèle (environ 512 tokens) sont tronquées, donc la fin des textes très
# longs est ignorée par le modèle.
res = analyseur(echantillon["text"].tolist(), truncation=True, batch_size=16)

trad = {"positive": "positif", "negative": "négatif", "neutral": "neutre"}
echantillon["sentiment_predit"] = [trad[r["label"]] for r in res]

pd.crosstab(echantillon["polarite"], echantillon["sentiment_predit"],
            rownames=["vrai"], colnames=["prédit"])

Le modèle n'a **jamais vu** notre corpus et s'en sort déjà bien. Avantage : aucun entraînement. Limite : on ne contrôle ni ce qu'il a appris, ni ses catégories (il ajoute un `neutre` qui n'existe pas chez nous).

### Hack Time

Compare les prédictions du modèle aux **vraies étiquettes** (`polarite`) sur un échantillon, et repère les cas où ils sont en **désaccord** : ce sont les erreurs du modèle. Lis quelques-uns de ces textes : pourquoi le modèle se trompe-t-il ?

Ce modèle de sentiment a été entraîné sur certaines données et peut porter des **biais** : il lit parfois mal l'ironie, la négation ou un éloge propre à un domaine (ce qui est positif pour un film ne l'est pas forcément ailleurs). Sa sortie est une **mesure entachée d'erreur**, pas une vérité de terrain. Pour la recherche, cela change tout : avant de t'y fier, tu la **valides** contre un échantillon codé à la main pour estimer son taux d'erreur.

**Défi optionnel :** code à la main une dizaine de critiques que le modèle a mal classées, et calcule sur cet échantillon le taux d'accord entre le modèle et ton codage.

In [ ]:
# Votre code ici
# Piste : isole les lignes où le modèle et la vraie étiquette diffèrent.
desaccords = echantillon[echantillon["polarite"] != echantillon["sentiment_predit"]]
# À toi : combien y en a-t-il ? Affiche-en quelques-unes et lis les textes.


## 4.3 Application 3 : Classification supervisée (machine learning)

*On entraîne nous-mêmes un modèle.*

**À quoi ça sert ?** C'est le cœur du **codage automatisé**. Si tu as une grille de codage et quelques centaines de textes codés à la main, tu apprends à la machine à coder le reste : trier 200 000 commentaires Facebook, classer des décisions de justice par type, ranger des articles par thème. On démultiplie un travail de codage manuel, avec une **mesure de fiabilité** (accuracy, matrice de confusion) qu'un codage purement humain n'offre pas toujours.

**Comment ?** On vectorise les critiques en **TF-IDF** (vu au notebook 1), puis on entraîne une **régression logistique** à prédire la polarité : exactement le cadre du machine learning supervisé vu en J2, mais avec du texte en entrée.

![Schéma classification supervisée](https://raw.githubusercontent.com/mickaeltemporao/lillelms/refine-notebooks-j2-j3-matin/ressources/j3/1_matin/img/m4-classification.png)

*Classification supervisée : on apprend sur des textes codés, la machine code le reste.*

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

tfidf = TfidfVectorizer(min_df=5, stop_words=list(STOP_FR))
X_train = tfidf.fit_transform(train["text"])
X_test = tfidf.transform(test["text"])

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, train["polarite"])

pred = clf.predict(X_test)
print("Accuracy sur le test :", round(accuracy_score(test["polarite"], pred), 3))

ConfusionMatrixDisplay(
    confusion_matrix(test["polarite"], pred, labels=clf.classes_),
    display_labels=clf.classes_,
).plot(cmap="Blues"); plt.show()

Comme c'est un modèle **linéaire**, on peut lire ses coefficients : les mots qui poussent le plus vers « positif » ou « négatif ». On retrouve l'analyse de l'application 1, mais cette fois **apprise** pour maximiser la prédiction.

In [ ]:
vocab = np.array(tfidf.get_feature_names_out())
coef = clf.coef_[0]   # classes_ = ['négatif', 'positif'] -> coef pour 'positif'
ordre = coef.argsort()

print("Vers POSITIF :", ", ".join(vocab[ordre[::-1][:15]]))
print("Vers NÉGATIF :", ", ".join(vocab[ordre[:15]]))

### Hack Time

- Écris ta propre critique et fais-la prédire : `clf.predict(tfidf.transform(["..."]))`.
- Essaie une phrase ironique (« génial, j'ai adoré m'ennuyer ») : le modèle se laisse-t-il piéger ?

**Défi optionnel :** ajoute `ngram_range=(1, 2)` au `TfidfVectorizer`, réentraîne, et regarde si l'accuracy sur le test progresse.

In [ ]:
# Votre code ici

## 4.4 Application 4 : Recherche sémantique

*On passe aux embeddings denses (les mêmes que SBERT au notebook 1).*

**À quoi ça sert ?** Explorer un grand corpus **par le sens**, sans deviner les mots exacts employés : retrouver tous les passages d'entretiens qui parlent de précarité même quand le mot n'apparaît pas, repérer dans des règlements d'urbanisme (PLU) des dispositions équivalentes formulées différemment, ou se constituer un sous-corpus thématique pour une lecture qualitative. C'est aussi la brique de base des moteurs de recherche modernes.

**Comment ?** On encode chaque critique avec un modèle multilingue de phrases, puis on classe les critiques par **similarité cosinus** avec une requête en langage naturel. Une critique qui parle d'un « mauvais jeu d'acteur » ressort même si elle n'emploie pas ces mots exacts.

![Schéma recherche sémantique](https://raw.githubusercontent.com/mickaeltemporao/lillelms/refine-notebooks-j2-j3-matin/ressources/j3/1_matin/img/m4-search.png)

*Recherche sémantique : requête et textes sont des vecteurs, on renvoie les plus proches.*

In [ ]:
from sentence_transformers import SentenceTransformer, util

modele = SentenceTransformer("distiluse-base-multilingual-cased-v2")
corpus = test["text"].tolist()
emb_corpus = modele.encode(corpus, show_progress_bar=False, batch_size=64)

def rechercher(requete, k=3):
    emb = modele.encode([requete])
    score = util.cos_sim(emb, emb_corpus).numpy().ravel()
    for i in score.argsort()[::-1][:k]:
        print(f"[{score[i]:.2f}] ({test['polarite'].iloc[i]}) {corpus[i][:200]}...\n")

rechercher("un jeu d'acteur décevant et un scénario plat")

## Récap module 4

| # | Application | Outil principal | Supervisé ? | Difficulté |
|---|---|---|---|---|
| 1 | Nuage de mots / mots-clés | `CountVectorizer`, `wordcloud` | non | ★ |
| 2 | Analyse de sentiment | `transformers` (pré-entraîné) | non | ★★ |
| 3 | Classification | TF-IDF + `LogisticRegression` | **oui** | ★★★ |
| 4 | Recherche sémantique | `sentence-transformers` | non | ★★★ |

Du simple comptage de mots à la recherche par le sens, le text mining offre une **boîte à outils** pour explorer un corpus textuel.

> **Un dernier réflexe** : ces outils ne sont **pas neutres**. Un modèle hérite des biais de ses données d'entraînement (genre, origine, époque) et notre corpus lui-même est déséquilibré. Documenter le corpus et tester la robustesse fait partie de la méthode.

→ Direction l'**après-midi** : on quitte le texte pour les données **audio et visuelles** (transcription, analyse d'images).